# V5 Phase 1 pretraining + Phase 4 CJPE supervised teacher and knowledge distillation

This notebook keeps the uploaded V5 decoder-only model and its from-scratch language-model training workflow, corrects the model-size calculation/configuration, then adds a separate Phase 4 case-outcome classification experiment. It trains the V5 teacher from scratch, then distills its Phase 4 decision distribution into the existing Phase 1 V1 checkpoint. The task head is new for both models.

The uploaded 12-layer, 768-wide settings instantiate about 97M parameters; a permissive 50–300M assertion did not support the stated 130M target. The revised 12-layer, 1024-wide configuration is checked from the instantiated PyTorch model and yields 164,090,864 unique parameters (25k vocabulary with tied input/output embeddings).

**No score is promised.** The notebook selects checkpoints and thresholds with the development set only, and never uses test labels. The full V5 pretraining run is compute-heavy; choose a Kaggle GPU and allow a long session.

**Run order:** attach the listed Kaggle inputs, restart the session, and use **Run All** from the first cell. This notebook first performs Phase 1 V5 pretraining, then Phase 4 teacher training and distillation; downstream cells depend on files and variables created earlier.


In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    login(HF_TOKEN, add_to_git_credential=False)
    print("Hugging Face token loaded from Kaggle Secrets.")
except Exception as e:
    HF_TOKEN = None
    print("No HF_TOKEN secret found; attempting public dataset access.", type(e).__name__)


## 2. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 3. Configuration

#### this is sample configuration, you are free to modify any parameter to train the best performing model

In [ ]:
#VERSION 1: DECODER ONLY MODEL; using GQA+ROPE+RMSNORM

import os
import random
import json
import time
import math
import zipfile
import shutil
import torch
import numpy as np

SEED =42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("Basic setup loaded.")
print(f"Random seed: {SEED}")

In [ ]:
#DATASET
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

#for this version i am using just the supreme court training files
TRAIN_FILE = "Phase_1_Pretraining/Train/train_supreme_court.json"
TEST_FILE ="Phase_1_Pretraining/Test/test.json"

#for this version i was not sure of the computation timing so taking just 100M tokens
TRAIN_TOKENS = 135_000_000
VAL_TOKENS = 5_000_000

print("Dataset settings loaded.")
print("Training file:", TRAIN_FILE)
print("Training tokens:", f"{TRAIN_TOKENS:,}")
print("Validation tokens:", f"{VAL_TOKENS:,}")

In [ ]:
# Context length is set to 512 to reduce the activation footprint of a 160M+ model on Kaggle.
MODEL_CONTEXT_LENGTH=512
TRAIN_BLOCK_SIZE=MODEL_CONTEXT_LENGTH
print("Context and training block:",MODEL_CONTEXT_LENGTH)


In [ ]:
# V5 correction: 12 layers x width 1024, with 16 query heads and 8 KV heads.
# This materially exceeds 130M parameters with the 25k tied-embedding vocabulary.
N_LAYER=12
N_EMBD=1024
N_HEAD=16
N_KV_HEAD=8
FFN_MULTIPLIER=4
FFN_TYPE="SwiGLU"
DROPOUT=0.0
USE_ROPE=True
USE_RMSNORM=True
print("V5 architecture:", N_LAYER, "layers | width", N_EMBD, "| heads", N_HEAD, "/", N_KV_HEAD)


### Why these V5 settings

- **12 layers, width 1,024:** raises model capacity while keeping depth familiar. The old uploaded configuration (12 × 768, 25k tied vocabulary) is **97,134,336** parameters in its implemented SwiGLU architecture. The corrected settings are **164,090,864**, counted from the actual instantiated module.
- **16 query heads / 8 key-value heads:** head width remains 64, and each pair of query heads shares a KV head. This keeps attention dimensions consistent and reduces KV storage compared with full multi-head attention.
- **SwiGLU and tied embeddings:** retained from the uploaded V5 implementation; the input token embedding and output language-model projection share one parameter matrix, so the vocabulary matrix is counted once.
- **512-token context:** shorter than the uploaded 1,024 setting to fit the wider model’s activations on a Kaggle GPU. The task stage also uses 512-token windows, so its inputs fit the pretrained context directly.
- **135M training tokens / 5M validation tokens:** preserves the uploaded training budget. The validation cache is sampled at fixed, evenly spaced positions for bounded evaluation; it is a monitoring estimate, not a full-corpus perplexity calculation.
- **Microbatch 1 × accumulation 32:** gives an effective 16,384 tokens per optimizer update at length 512 while limiting the memory used by each forward/backward pass. BF16 is used when supported, otherwise FP16; activation checkpointing trades extra compute for lower activation memory.
- **Learning rate 1.5e-4, 300-step warmup, cosine decay, clip 1.0:** conservative settings for scratch training at this scale, with gradual startup and bounded gradient updates. The actual run history must determine whether the budget and schedule were adequate.



In [ ]:
# Conservative per-step memory for Kaggle GPUs; accumulation preserves the effective token batch.
PER_DEVICE_BATCH_SIZE=1
GRADIENT_ACCUMULATION_STEPS=32
LEARNING_RATE=1.5e-4
WEIGHT_DECAY=0.1
WARMUP_STEPS=300
MAX_GRAD_NORM=1.0
EVAL_EVERY_STEPS=500
LOG_EVERY_STEPS=50
SAVE_EVERY_STEPS=500
MAX_CHECKPOINTS_TO_KEEP=2
USE_AMP=torch.cuda.is_available()
USE_GRADIENT_CHECKPOINTING=True
print("Microbatch / accumulation:",PER_DEVICE_BATCH_SIZE,GRADIENT_ACCUMULATION_STEPS)


In [ ]:
from pathlib import Path
OUT=Path(OUTPUT_DIR)
# Output paths. Generation settings from the source notebook are omitted because Phase 4 is classification.
OUTPUT_DIR="/kaggle/working/phase4_v5_kd"
CHECKPOINT_DIR=os.path.join(OUTPUT_DIR,"checkpoints")
CACHE_DIR=os.path.join(OUTPUT_DIR,"cache")
LOG_DIR=os.path.join(OUTPUT_DIR,"logs")
MODEL_OUTPUT_DIR=os.path.join(OUTPUT_DIR,"final_model")
CONFIG_PATH=os.path.join(OUTPUT_DIR,"config.json")
for directory in (OUTPUT_DIR,CHECKPOINT_DIR,CACHE_DIR,LOG_DIR,MODEL_OUTPUT_DIR): os.makedirs(directory,exist_ok=True)


In [ ]:
# Record only settings that are active in this pretraining + CJPE workflow.
experiment_config={"seed":SEED,"dataset":HF_DATASET_REPO,"train_file":TRAIN_FILE,"train_tokens":TRAIN_TOKENS,"val_tokens":VAL_TOKENS,"context_length":MODEL_CONTEXT_LENGTH,"n_layer":N_LAYER,"n_embd":N_EMBD,"n_head":N_HEAD,"n_kv_head":N_KV_HEAD,"ffn_multiplier":FFN_MULTIPLIER,"ffn_type":FFN_TYPE,"batch_size":PER_DEVICE_BATCH_SIZE,"gradient_accumulation_steps":GRADIENT_ACCUMULATION_STEPS,"learning_rate":LEARNING_RATE,"weight_decay":WEIGHT_DECAY,"warmup_steps":WARMUP_STEPS,"max_grad_norm":MAX_GRAD_NORM,"use_amp":USE_AMP,"gradient_checkpointing":USE_GRADIENT_CHECKPOINTING}
with open(CONFIG_PATH,"w",encoding="utf-8") as f: json.dump(experiment_config,f,indent=2)
print(json.dumps(experiment_config,indent=2))


## 4. Load Tokenizer

In [ ]:
#tokenizer from A1
import os
import json

BPE_FILENAME = "bpe_en_25k.json"
bpe_candidates = []

for root, dirs, files in os.walk("/kaggle/input"):
    if BPE_FILENAME in files:
        bpe_candidates.append(os.path.join(root, BPE_FILENAME))

if len(bpe_candidates) == 0:
    raise FileNotFoundError(
        f"{BPE_FILENAME} was not found inside /kaggle/input/. "
        "Make sure the BPE-25k dataset is attached."
    )

BPE_PATH = bpe_candidates[0]

print("Tokenizer found:", BPE_PATH)


# Load the vocabulary and learned BPE merges.
with open(BPE_PATH, "r", encoding="utf-8") as f:
    bpe_data = json.load(f)

bpe_itos = bpe_data["itos"]
bpe_merges = bpe_data["merges"]
bpe_vocab_size = bpe_data["vocab_size"]

# Make sure we have loaded the tokenizer we actually want.
assert bpe_vocab_size == 25_000
assert len(bpe_itos) == 25_000

print("Vocabulary size:", bpe_vocab_size)
print("Number of merge rules:", len(bpe_merges))


# Token -> ID mapping
bpe_stoi = {
    token: idx
    for idx, token in enumerate(bpe_itos)
}
merge_to_id = {}

for merge_index, merge in enumerate(bpe_merges):
    left_token, right_token = merge

    left_id = bpe_stoi[left_token]
    right_id = bpe_stoi[right_token]

    new_token_id = 261 + merge_index
    merge_to_id[(left_id, right_id)] = new_token_id

PAD_TOKEN_ID = bpe_stoi["<pad>"]
UNK_TOKEN_ID = bpe_stoi["<unk>"]
BOS_TOKEN_ID = bpe_stoi["<s>"]
EOS_TOKEN_ID = bpe_stoi["</s>"]
WORD_BOUNDARY_ID = bpe_stoi["<wb>"]

print("\nSpecial token IDs:")
print("PAD:", PAD_TOKEN_ID)
print("UNK:", UNK_TOKEN_ID)
print("BOS:", BOS_TOKEN_ID)
print("EOS:", EOS_TOKEN_ID)
print("WORD BOUNDARY:", WORD_BOUNDARY_ID)

In [ ]:
#Tokenizer Sanity Check
#Before using the tokenizer on the full legal corpus,let's check a small example and make sure encoding and decoding work correctly.

class ByteLevelBPE25k:
    def __init__(self, itos, stoi, merge_to_id, merges):
        self.itos = itos
        self.stoi = stoi
        self.merge_to_id = merge_to_id
        self.merges = merges
        self.vocab_size = len(itos)

        self.pad_token_id = stoi["<pad>"]
        self.unk_token_id = stoi["<unk>"]
        self.bos_token_id = stoi["<s>"]
        self.eos_token_id = stoi["</s>"]
        self.word_boundary_id = stoi["<wb>"]

    def _encode_word(self, word):
        # Start with the UTF-8 bytes of the word.
        sequence = [
            4 + byte
            for byte in word.encode("utf-8")
        ]

        # Our BPE representation keeps a word-boundary token.
        sequence.append(self.word_boundary_id)

        while len(sequence) > 1:
            best_position = None
            best_token_id = None

            for i in range(len(sequence) - 1):
                pair = (sequence[i], sequence[i + 1])
                new_token_id = self.merge_to_id.get(pair)

                if new_token_id is None:
                    continue

                if (
                    best_token_id is None
                    or new_token_id < best_token_id
                ):
                    best_token_id = new_token_id
                    best_position = i

            if best_position is None:
                break

            i = best_position
            sequence = (
                sequence[:i]
                + [best_token_id]
                + sequence[i + 2:]
            )

        return sequence

    def encode(self, text, add_bos=False, add_eos=False):
        # The original tokenizer was trained word-by-word.
        words = text.split()

        token_ids = []

        if add_bos:
            token_ids.append(self.bos_token_id)

        for word in words:
            token_ids.extend(
                self._encode_word(word)
            )

        if add_eos:
            token_ids.append(self.eos_token_id)

        return token_ids

    def _token_to_bytes(self, token_id):
        token = self.itos[token_id]

        # Raw byte token.
        if token.startswith("<0x") and token.endswith(">"):
            return bytes([int(token[3:-1], 16)])

        if token == "<wb>":
            return b" "

        if token in {"<pad>", "<unk>", "<s>", "</s>"}:
            return b""

        # Learned BPE token.
        if token.startswith("<bpe_"):
            merge_index = token_id - 261

            if 0 <= merge_index < len(self.merges):
                left_token, right_token = self.merges[merge_index]

                left_id = self.stoi[left_token]
                right_id = self.stoi[right_token]

                return (
                    self._token_to_bytes(left_id)
                    + self._token_to_bytes(right_id)
                )

        return b""

    def decode(self, token_ids):
        output = bytearray()

        for token_id in token_ids:
            if not 0 <= int(token_id) < self.vocab_size:
                continue

            output.extend(
                self._token_to_bytes(int(token_id))
            )

        return bytes(output).decode(
            "utf-8",
            errors="replace"
        )


# Creating the tokenizer.
tokenizer = ByteLevelBPE25k(
    itos=bpe_itos,
    stoi=bpe_stoi,
    merge_to_id=merge_to_id,
    merges=bpe_merges
)


# Test it on a small legal-style sentence.
sample_text = (
    "The Supreme Court of India has jurisdiction "
    "over constitutional matters."
)

sample_tokens = tokenizer.encode(sample_text)
decoded_text = tokenizer.decode(sample_tokens)

print("Original text:")
print(sample_text)
print("\nToken IDs:")
print(sample_tokens)
print("\nNumber of tokens:", len(sample_tokens))
print("\nDecoded text:")
print(decoded_text)
print("\nVocabulary size:", tokenizer.vocab_size)
assert tokenizer.vocab_size == 25_000
assert len(sample_tokens) > 0
print("\nTokenizer sanity check passed.")

## 5. Download Training Data From Hugging Face

In [ ]:
from huggingface_hub import hf_hub_download

train_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TRAIN_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Training file:")
print(train_file_path)

## 6. Create Train and Validation Tokens Set

In [ ]:
# 6. Create Train and Validation Token Cache
import os
import json
import random
import time
import numpy as np

CACHE_DIR = os.path.join(
    OUTPUT_DIR,
    "token_cache"
)

os.makedirs(CACHE_DIR, exist_ok=True)

TRAIN_TOKEN_PATH = os.path.join(
    CACHE_DIR,
    "train_tokens.bin"
)

VAL_TOKEN_PATH = os.path.join(
    CACHE_DIR,
    "val_tokens.bin"
)

TOKEN_CACHE_META = os.path.join(
    CACHE_DIR,
    "cache_metadata.json"
)

# Load the official training corpus
with open(train_file_path, "r", encoding="utf-8") as f:
    records = json.load(f)

print("Number of records:", len(records))

# Resume from an existing cache if possible

train_count = 0
val_count = 0
start_record = 0

if (
    os.path.exists(TRAIN_TOKEN_PATH)
    and os.path.exists(VAL_TOKEN_PATH)
    and os.path.exists(TOKEN_CACHE_META)
):
    with open(TOKEN_CACHE_META, "r", encoding="utf-8") as f:
        cache_meta = json.load(f)

    # Make sure this cache belongs to the same experiment.
    if (
        cache_meta.get("train_tokens") == TRAIN_TOKENS
        and cache_meta.get("val_tokens") == VAL_TOKENS
        and cache_meta.get("vocab_size") == tokenizer.vocab_size
        and cache_meta.get("seed") == SEED
    ):
        start_record = cache_meta["next_record"]
        train_count = cache_meta["train_count"]
        val_count = cache_meta["val_count"]

        print("Existing token cache found.")
        print("Resuming from record:", start_record)
        print("Train tokens already cached:", f"{train_count:,}")
        print("Validation tokens already cached:", f"{val_count:,}")

    else:
        print("Existing cache does not match the current settings.")
        print("Starting a fresh token cache.")

        os.remove(TRAIN_TOKEN_PATH)
        os.remove(VAL_TOKEN_PATH)
        os.remove(TOKEN_CACHE_META)

else:
    print("No existing token cache found.")
    print("Starting tokenization from the beginning.")

# Open cache files in append mode

train_file = open(
    TRAIN_TOKEN_PATH,
    "ab"
)

val_file = open(
    VAL_TOKEN_PATH,
    "ab"
)

# Tokenize the corpus
# We keep the same record-level random split used earlier:roughly 95% for training and 5% for validation.The seed and record index are saved, so the split stays fixed when the process is resumed.

random.seed(SEED)

# Recreate the same random decisions for records that were
# already processed before an interruption.
for _ in range(start_record):
    random.random()

start_time = time.time()

for record_index in range(start_record, len(records)):

    # Stop once both requested token budgets are reached.
    if (
        train_count >= TRAIN_TOKENS
        and val_count >= VAL_TOKENS
    ):
        break

    text = records[record_index]["text"]

    # Use our custom BPE-25k tokenizer.
    tokens = tokenizer.encode(
        text,
        add_bos=False,
        add_eos=False
    )

    # Decide whether this record belongs to validation.
    is_validation = random.random() < 0.05

    if is_validation and val_count < VAL_TOKENS:

        remaining = VAL_TOKENS - val_count
        tokens = tokens[:remaining]

        if tokens:
            token_array = np.asarray(
                tokens,
                dtype=np.uint32
            )

            val_file.write(
                token_array.tobytes()
            )

            val_count += len(token_array)

    elif not is_validation and train_count < TRAIN_TOKENS:

        remaining = TRAIN_TOKENS - train_count
        tokens = tokens[:remaining]

        if tokens:
            token_array = np.asarray(
                tokens,
                dtype=np.uint32
            )

            train_file.write(
                token_array.tobytes()
            )

            train_count += len(token_array)


    # Save progress periodically so an interruption does notforce us to start tokenization from zero.
    if (
        (record_index + 1) % 100 == 0
        or train_count >= TRAIN_TOKENS
        or val_count >= VAL_TOKENS
    ):

        train_file.flush()
        val_file.flush()

        cache_meta = {
            "train_tokens": TRAIN_TOKENS,
            "val_tokens": VAL_TOKENS,
            "train_count": train_count,
            "val_count": val_count,
            "next_record": record_index + 1,
            "vocab_size": tokenizer.vocab_size,
            "seed": SEED
        }

        with open(
            TOKEN_CACHE_META,
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                cache_meta,
                f,
                indent=2
            )

    # Print progress every 100 records.
    if (record_index + 1) % 100 == 0:

        elapsed = time.time() - start_time

        print(
            f"Record {record_index + 1:,}/{len(records):,} | "
            f"Train: {train_count:,}/{TRAIN_TOKENS:,} | "
            f"Val: {val_count:,}/{VAL_TOKENS:,} | "
            f"Time: {elapsed / 60:.1f} min"
        )


train_file.flush()
val_file.flush()

train_file.close()
val_file.close()

# Final checks

assert train_count == TRAIN_TOKENS, (
    f"Expected {TRAIN_TOKENS:,} train tokens, "
    f"got {train_count:,}"
)

assert val_count == VAL_TOKENS, (
    f"Expected {VAL_TOKENS:,} validation tokens, "
    f"got {val_count:,}"
)

print("\nTokenization complete.")
print("Train tokens:", f"{train_count:,}")
print("Validation tokens:", f"{val_count:,}")
print("Train cache:", TRAIN_TOKEN_PATH)
print("Validation cache:", VAL_TOKEN_PATH)

## 7. Create Token Blocks

In [ ]:
# Create Dataset From Cached Tokens

import numpy as np
import torch
from torch.utils.data import Dataset

class LegalTokenDataset(Dataset):
    def __init__(
        self,
        token_path,
        num_tokens,
        block_size,
        stride=1
    ):
        self.token_path = token_path
        self.num_tokens = num_tokens
        self.block_size = block_size
        self.stride = stride

        # The file stays on disk and is accessed only when
        # a particular training example is requested.
        self.tokens = np.memmap(
            token_path,
            dtype=np.uint32,
            mode="r",
            shape=(num_tokens,)
        )

    def __len__(self):
        return 1 + (
            self.num_tokens
            - self.block_size
            - 1
        ) // self.stride

    def __getitem__(self, index):
        start = index * self.stride
        end = start + self.block_size + 1

        tokens = self.tokens[start:end].astype(
            np.int64
        )

        input_ids = torch.from_numpy(
            tokens[:-1]
        )

        labels = torch.from_numpy(
            tokens[1:]
        )

        return {
            "input_ids": input_ids,
            "labels": labels
        }

train_dataset = LegalTokenDataset(
    token_path=TRAIN_TOKEN_PATH,
    num_tokens=TRAIN_TOKENS,
    block_size=TRAIN_BLOCK_SIZE,
    stride=TRAIN_BLOCK_SIZE
)

val_dataset = LegalTokenDataset(
    token_path=VAL_TOKEN_PATH,
    num_tokens=VAL_TOKENS,
    block_size=TRAIN_BLOCK_SIZE,
    stride=TRAIN_BLOCK_SIZE
)

print("Train examples:", f"{len(train_dataset):,}")
print("Validation examples:", f"{len(val_dataset):,}")
print("Sequence length:", TRAIN_BLOCK_SIZE)

## 8. Create Datasets

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=PER_DEVICE_BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=PER_DEVICE_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

print("Train batches per epoch:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Batch size:", PER_DEVICE_BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)

In [ ]:
#SANITY CHECK
batch = next(iter(train_loader))

print("Input shape:", batch["input_ids"].shape)
print("Label shape:", batch["labels"].shape)

print(
    "Minimum token ID:",
    batch["input_ids"].min().item()
)

print(
    "Maximum token ID:",
    batch["input_ids"].max().item()
)

assert batch["input_ids"].shape == (
    PER_DEVICE_BATCH_SIZE,
    TRAIN_BLOCK_SIZE
)
assert batch["labels"].shape == (
    PER_DEVICE_BATCH_SIZE,
    TRAIN_BLOCK_SIZE
)
assert batch["input_ids"].min() >= 0
assert batch["input_ids"].max() < tokenizer.vocab_size

print("\nDataLoader sanity check passed.")

## 9. Define The Model

In [ ]:
# 9. V1 Decoder-Only Transformer
# This is our first architecture built from scratch.
# V1 uses:
# - RMSNorm
# - RoPE
# - Grouped Query Attention
# - Causal self-attention
# - SwiGLU FFN (the implementation is SwiGLU despite the old comment)
# - Residual connections
# - KV cache for autoregressive generation

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


# ---------------------------------------------------------
# RMSNorm
# ---------------------------------------------------------

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(
            dim=-1,
            keepdim=True
        )

        x = x * torch.rsqrt(
            rms + self.eps
        )

        return self.weight * x


# ---------------------------------------------------------
# Rotary Positional Embeddings
# ---------------------------------------------------------

class RotaryEmbedding(nn.Module):
    def __init__(
        self,
        head_dim,
        max_seq_len,
        base=10000
    ):
        super().__init__()

        self.head_dim = head_dim
        self.max_seq_len = max_seq_len

        inv_freq = 1.0 / (
            base ** (
                torch.arange(
                    0,
                    head_dim,
                    2
                ).float()
                / head_dim
            )
        )

        self.register_buffer(
            "inv_freq",
            inv_freq,
            persistent=False
        )

        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        positions = torch.arange(
            seq_len,
            device=self.inv_freq.device
        ).float()

        freqs = torch.outer(
            positions,
            self.inv_freq
        )

        self.register_buffer(
            "cos_cached",
            freqs.cos(),
            persistent=False
        )

        self.register_buffer(
            "sin_cached",
            freqs.sin(),
            persistent=False
        )

    def forward(
        self,
        q,
        k,
        position_offset=0
    ):
        seq_len = q.size(-2)

        required_len = (
            position_offset + seq_len
        )

        if required_len > self.max_seq_len:
            self._build_cache(required_len)
            self.max_seq_len = required_len

        cos = self.cos_cached[
            position_offset:
            position_offset + seq_len
        ]

        sin = self.sin_cached[
            position_offset:
            position_offset + seq_len
        ]

        cos = cos.unsqueeze(0).unsqueeze(0)
        sin = sin.unsqueeze(0).unsqueeze(0)

        q = self._apply_rotary(
            q,
            cos,
            sin
        )

        k = self._apply_rotary(
            k,
            cos,
            sin
        )

        return q, k

    @staticmethod
    def _apply_rotary(x, cos, sin):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]

        rotated = torch.stack(
            (-x2, x1),
            dim=-1
        ).flatten(-2)

        cos_full = torch.repeat_interleave(
            cos,
            2,
            dim=-1
        )

        sin_full = torch.repeat_interleave(
            sin,
            2,
            dim=-1
        )

        return (
            x * cos_full
            + rotated * sin_full
        )


# ---------------------------------------------------------
# Grouped Query Attention
# ---------------------------------------------------------

class GQAAttention(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        n_kv_head,
        max_seq_len,
        dropout=0.0
    ):
        super().__init__()

        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0

        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.head_dim = n_embd // n_head

        self.kv_repeat = (
            n_head // n_kv_head
        )

        self.q_proj = nn.Linear(
            n_embd,
            n_head * self.head_dim,
            bias=False
        )

        self.k_proj = nn.Linear(
            n_embd,
            n_kv_head * self.head_dim,
            bias=False
        )

        self.v_proj = nn.Linear(
            n_embd,
            n_kv_head * self.head_dim,
            bias=False
        )

        self.out_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.rope = RotaryEmbedding(
            head_dim=self.head_dim,
            max_seq_len=max_seq_len
        )

        self.dropout = dropout

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False
    ):
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B,
            T,
            self.n_head,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.n_kv_head,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.n_kv_head,
            self.head_dim
        ).transpose(1, 2)

        past_len = 0

        if past_key_value is not None:
            past_k, past_v = past_key_value
            past_len = past_k.size(-2)

        # Apply RoPE to the new Q and K positions.
        q, k = self.rope(
            q,
            k,
            position_offset=past_len
        )

        # Add the new K/V to the cached K/V during generation.
        if past_key_value is not None:
            k = torch.cat(
                [past_k, k],
                dim=-2
            )

            v = torch.cat(
                [past_v, v],
                dim=-2
            )

        present = None

        if use_cache:
            present = (k, v)

        # GQA: each KV head is shared by a group of query heads.
        if self.kv_repeat > 1:
            k = k.repeat_interleave(
                self.kv_repeat,
                dim=1
            )

            v = v.repeat_interleave(
                self.kv_repeat,
                dim=1
            )

        # During training, each token can only attend to
        # itself and tokens before it.
        #
        # IMPORTANT:
        # For scaled_dot_product_attention, True means
        # "this position is allowed to attend".
        if past_key_value is None:
            attn_mask = torch.tril(
                torch.ones(
                    T,
                    T,
                    device=x.device,
                    dtype=torch.bool
                )
            )
        else:
            # During cached decoding, all cached positions
            # are valid keys for the current token.
            attn_mask = None

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=(
                self.dropout
                if self.training
                else 0.0
            ),
            is_causal=False
        )

        y = y.transpose(
            1,
            2
        ).contiguous()

        y = y.view(
            B,
            T,
            C
        )

        y = self.out_proj(y)

        return y, present

# ---------------------------------------------------------
# Parameter-Matched SwiGLU FFN
# ---------------------------------------------------------

class FeedForward(nn.Module):
    def __init__(
        self,
        n_embd,
        multiplier=4,
        dropout=0.0
    ):
        super().__init__()

        # Parameter-matched SwiGLU hidden size.
        # Standard FFN has approximately:
        #   2 * n_embd * (4 * n_embd)
        #
        # SwiGLU has three projections:
        #   gate, up, down
        #
        # Using 2/3 scaling keeps the parameter count
        # approximately matched.

        hidden_dim = int(
            (2.0 / 3.0)
            * n_embd
            * multiplier
        )

        self.gate_proj = nn.Linear(
            n_embd,
            hidden_dim,
            bias=True
        )

        self.up_proj = nn.Linear(
            n_embd,
            hidden_dim,
            bias=True
        )

        self.down_proj = nn.Linear(
            hidden_dim,
            n_embd,
            bias=True
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):

        gate = F.silu(
            self.gate_proj(x)
        )

        up = self.up_proj(x)

        x = gate * up

        x = self.down_proj(x)

        x = self.dropout(x)

        return x

# ---------------------------------------------------------
# Transformer Block
# ---------------------------------------------------------

class TransformerBlock(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        n_kv_head,
        max_seq_len,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.norm1 = RMSNorm(n_embd)

        self.attention = GQAAttention(
            n_embd=n_embd,
            n_head=n_head,
            n_kv_head=n_kv_head,
            max_seq_len=max_seq_len,
            dropout=dropout
        )

        self.norm2 = RMSNorm(n_embd)

        self.ffn = FeedForward(
            n_embd=n_embd,
            multiplier=ffn_multiplier,
            dropout=dropout
        )

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False
    ):
        # Pre-norm attention followed by a residual connection.
        attn_output, present = self.attention(
            self.norm1(x),
            past_key_value=past_key_value,
            use_cache=use_cache
        )

        x = x + attn_output

        # Pre-norm FFN followed by a residual connection.
        x = x + self.ffn(
            self.norm2(x)
        )

        return x, present


# ---------------------------------------------------------
# V1 Language Model
# ---------------------------------------------------------

class V1LegalLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embd,
        n_layer,
        n_head,
        n_kv_head,
        max_seq_len,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.n_embd = n_embd

        self.token_embedding = nn.Embedding(
            vocab_size,
            n_embd
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                n_embd=n_embd,
                n_head=n_head,
                n_kv_head=n_kv_head,
                max_seq_len=max_seq_len,
                ffn_multiplier=ffn_multiplier,
                dropout=dropout
            )
            for _ in range(n_layer)
        ])

        self.final_norm = RMSNorm(
            n_embd
        )

        # Tie the LM head to the token embedding weights.
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False
        )

        self.lm_head.weight = (
            self.token_embedding.weight
        )

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False,
        return_hidden=False
    ):
        x = self.token_embedding(
            input_ids
        )

        presents = [] if use_cache else None

        for layer_index, block in enumerate(
            self.blocks
        ):
            past = None

            if past_key_values is not None:
                past = past_key_values[
                    layer_index
                ]

            if self.training and getattr(self, "gradient_checkpointing", False) and not use_cache:
                x = checkpoint(lambda h, layer=block: layer(h, past_key_value=None, use_cache=False)[0], x, use_reentrant=False)
                present = None
            else:
                x, present = block(
                    x,
                    past_key_value=past,
                    use_cache=use_cache
                )

            if use_cache:
                presents.append(
                    present
                )

        x = self.final_norm(x)
        if return_hidden:
            return x
        logits = self.lm_head(x)
        return logits, presents

In [ ]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
model = V1LegalLM(
    vocab_size=tokenizer.vocab_size,
    n_embd=N_EMBD,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    n_kv_head=N_KV_HEAD,
    max_seq_len=MODEL_CONTEXT_LENGTH,
    ffn_multiplier=FFN_MULTIPLIER,
    dropout=DROPOUT
).to(DEVICE)

print("V1 model created successfully.")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Layers: {N_LAYER}")
print(f"Embedding dimension: {N_EMBD}")
print(f"Attention heads: {N_HEAD}")
print(f"KV heads: {N_KV_HEAD}")
print(f"Context length: {MODEL_CONTEXT_LENGTH}")

In [ ]:
total_params=sum(p.numel() for p in model.parameters())
trainable_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Unique total parameters: {total_params:,} ({total_params/1e6:.3f}M)")
print(f"Trainable parameters: {trainable_params:,}")
assert total_params > 130_000_000, f"V5 must exceed 130M; got {total_params:,}"
print("Verified: actual instantiated V5 exceeds 130M parameters.")


In [ ]:
model.eval()

test_batch = next(iter(train_loader))

test_input_ids = test_batch["input_ids"][:2].to(DEVICE)
test_labels = test_batch["labels"][:2].to(DEVICE)

with torch.no_grad():
    test_logits, _ = model(
        test_input_ids,
        use_cache=False
    )

print("Input shape:", test_input_ids.shape)
print("Labels shape:", test_labels.shape)
print("Logits shape:", test_logits.shape)

expected_shape = (
    test_input_ids.shape[0],
    test_input_ids.shape[1],
    tokenizer.vocab_size
)

assert test_logits.shape == expected_shape
assert torch.isfinite(test_logits).all()

print("\nForward pass check passed.")

In [ ]:
model.train()
with torch.autocast(device_type="cuda",dtype=(torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16),enabled=USE_AMP):
    logits,_=model(test_input_ids,use_cache=False)
    loss=F.cross_entropy(logits.reshape(-1,tokenizer.vocab_size),test_labels.reshape(-1))
print("Initial loss:",float(loss.detach().cpu()))
assert torch.isfinite(loss)
model.zero_grad(set_to_none=True)


In [ ]:
#Causal Mask Test
# A decoder-only model must not allow a token to use
# information from future positions.We test this by changing a future token and checking that an earlier position gives the same output.

model.eval()

torch.manual_seed(SEED)

test_ids = torch.randint(
    low=0,
    high=tokenizer.vocab_size,
    size=(1, 16),
    device=DEVICE
)

# Make a copy and change only a future token.
modified_ids = test_ids.clone()
modified_ids[0, 10] = (
    modified_ids[0, 10] + 1
) % tokenizer.vocab_size

with torch.no_grad():
    logits_original, _ = model(
        test_ids,
        use_cache=False
    )

    logits_modified, _ = model(
        modified_ids,
        use_cache=False
    )

# Position 5 cannot see position 10.
earlier_difference = (
    logits_original[0, 5]
    - logits_modified[0, 5]
).abs().max().item()

# Position 10 can see itself, so changing it should affect
# its output in general.
current_difference = (
    logits_original[0, 10]
    - logits_modified[0, 10]
).abs().max().item()

print(
    f"Difference at earlier position: {earlier_difference:.10f}"
)

print(
    f"Difference at modified position: {current_difference:.10f}"
)

assert earlier_difference < 1e-5
assert current_difference > 1e-6

print("\nCausal mask check passed.")

In [ ]:
# KV Cache Equivalence Test
# During generation, KV caching should give the same results
# as processing the complete sequence normally.
# We compare:
# 1. Full-sequence forward pass
# 2. Token-by-token forward pass using the KV cache

model.eval()

torch.manual_seed(SEED)

cache_test_ids = torch.randint(
    low=0,
    high=tokenizer.vocab_size,
    size=(1, 16),
    device=DEVICE
)

with torch.no_grad():

    # Full sequence in one forward pass.
    full_logits, _ = model(
        cache_test_ids,
        use_cache=False
    )

    # Process one token at a time while keeping the
    # previous keys and values in the cache.
    past_key_values = None
    cached_logits = []

    for position in range(
        cache_test_ids.size(1)
    ):

        current_token = cache_test_ids[
            :, position:position + 1
        ]

        logits, past_key_values = model(
            current_token,
            past_key_values=past_key_values,
            use_cache=True
        )

        cached_logits.append(logits)

    cached_logits = torch.cat(
        cached_logits,
        dim=1
    )


# Compare full-sequence and cached outputs.
max_difference = (
    full_logits - cached_logits
).abs().max().item()

mean_difference = (
    full_logits - cached_logits
).abs().mean().item()

print(
    f"Maximum logit difference: {max_difference:.10f}"
)

print(
    f"Mean logit difference:    {mean_difference:.10f}"
)

assert torch.allclose(
    full_logits,
    cached_logits,
    atol=1e-4,
    rtol=1e-4
)

print("\nKV cache equivalence check passed.")

In [ ]:
# # Tiny Overfit Test
# # Sanity check

# model.train()

# tiny_input_ids = test_batch["input_ids"][:2].to(DEVICE)
# tiny_labels = test_batch["labels"][:2].to(DEVICE)

# tiny_optimizer = torch.optim.AdamW(
#     model.parameters(),
#     lr=1e-3,
#     weight_decay=0.0
# )

# initial_tiny_loss = None

# for step in range(100):

#     tiny_optimizer.zero_grad(
#         set_to_none=True
#     )

#     logits, _ = model(
#         tiny_input_ids,
#         use_cache=False
#     )

#     loss = F.cross_entropy(
#         logits.reshape(-1, tokenizer.vocab_size),
#         tiny_labels.reshape(-1)
#     )

#     if initial_tiny_loss is None:
#         initial_tiny_loss = loss.item()

#     loss.backward()

#     torch.nn.utils.clip_grad_norm_(
#         model.parameters(),
#         max_norm=1.0
#     )

#     tiny_optimizer.step()

#     if step == 0 or (step + 1) % 20 == 0:
#         print(
#             f"Step {step + 1:3d} | "
#             f"Loss: {loss.item():.6f}"
#         )

# final_tiny_loss = loss.item()

# print(
#     f"\nInitial tiny loss: {initial_tiny_loss:.6f}"
# )

# print(
#     f"Final tiny loss:   {final_tiny_loss:.6f}"
# )

# assert final_tiny_loss < initial_tiny_loss

# print("\nTiny overfit check passed.")

In [ ]:
# Fresh V1 model for the actual training run.

model = V1LegalLM(
    vocab_size=tokenizer.vocab_size,
    n_embd=N_EMBD,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    n_kv_head=N_KV_HEAD,
    max_seq_len=MODEL_CONTEXT_LENGTH,
    ffn_multiplier=FFN_MULTIPLIER,
    dropout=DROPOUT
).to(DEVICE)

print("Fresh V1 model created for training.")
print(
    f"Parameters: "
    f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M"
)

## 10. Training Arguments

In [ ]:
import os
import json
import math
import time

# Fresh optimizer for the freshly initialized model.
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    betas=(0.9, 0.95),
    weight_decay=WEIGHT_DECAY
)

# Number of tokens processed in one optimizer step.
TOKENS_PER_STEP = (
    PER_DEVICE_BATCH_SIZE
    * GRADIENT_ACCUMULATION_STEPS
    * TRAIN_BLOCK_SIZE
)

MAX_TRAIN_STEPS = math.ceil(
    TRAIN_TOKENS / TOKENS_PER_STEP
)

print(f"Tokens per optimizer step: {TOKENS_PER_STEP:,}")
print(f"Total training tokens: {TRAIN_TOKENS:,}")
print(f"Planned optimizer steps: {MAX_TRAIN_STEPS:,}")

# Cosine learning-rate schedule with warmup.
def get_learning_rate(step):
    if step < WARMUP_STEPS:
        return LEARNING_RATE * (
            (step + 1) / WARMUP_STEPS
        )

    progress = (
        step - WARMUP_STEPS
    ) / max(
        1,
        MAX_TRAIN_STEPS - WARMUP_STEPS
    )

    progress = min(max(progress, 0.0), 1.0)

    return LEARNING_RATE * 0.5 * (
        1.0 + math.cos(math.pi * progress)
    )

print("Training setup ready.")

In [ ]:
# Checkpoint Setup

CHECKPOINT_DIR = os.path.join(
    OUTPUT_DIR,
    "checkpoints"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

LATEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "latest.pt"
)

BEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "best.pt"
)

history = {
    "step": [],
    "train_loss": [],
    "val_loss": [],
    "val_perplexity": [],
    "learning_rate": [],
    "tokens_seen": []
}

best_val_loss = float("inf")

print("Checkpoint directory:", CHECKPOINT_DIR)

In [ ]:
@torch.no_grad()
def evaluate_model(max_batches=128):
    model.eval(); total_loss=0.0; total_tokens=0
    # Fixed, evenly spaced examples cover the complete validation token cache.
    n=min(max_batches,len(val_dataset))
    eval_indices=np.linspace(0,len(val_dataset)-1,n,dtype=np.int64)
    amp_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    for idx in eval_indices:
        batch=val_dataset[int(idx)]
        ids=batch["input_ids"].unsqueeze(0).to(DEVICE,non_blocking=True); labels=batch["labels"].unsqueeze(0).to(DEVICE,non_blocking=True)
        with torch.autocast(device_type="cuda",dtype=amp_dtype,enabled=USE_AMP):
            logits,_=model(ids,use_cache=False)
            loss=F.cross_entropy(logits.reshape(-1,tokenizer.vocab_size),labels.reshape(-1))
        total_loss+=float(loss)*labels.numel(); total_tokens+=labels.numel()
    model.train(); val_loss=total_loss/max(total_tokens,1)
    return val_loss,math.exp(min(val_loss,20))
print("Bounded validation ready (128 evenly spaced sequences from the validation cache).")


## 11. Training

In [ ]:
# Mixed-precision, gradient-checkpointed V5 training from random initialization.
if DEVICE.type!="cuda": raise RuntimeError("Select a Kaggle GPU accelerator before starting the 164M-parameter training run.")
scaler=torch.amp.GradScaler("cuda",enabled=(USE_AMP and not (torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)))
model.gradient_checkpointing=USE_GRADIENT_CHECKPOINTING
model.train(); global_step=0; tokens_seen=0; start_time=time.time(); optimizer.zero_grad(set_to_none=True)
train_iterator=iter(train_loader)
while global_step<MAX_TRAIN_STEPS:
    accumulated_loss=0.0
    for micro_step in range(GRADIENT_ACCUMULATION_STEPS):
        try: batch=next(train_iterator)
        except StopIteration:
            train_iterator=iter(train_loader); batch=next(train_iterator)
        ids=batch["input_ids"].to(DEVICE,non_blocking=True); labels=batch["labels"].to(DEVICE,non_blocking=True)
        amp_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
        with torch.autocast(device_type="cuda",dtype=amp_dtype,enabled=USE_AMP):
            logits,_=model(ids,use_cache=False)
            loss=F.cross_entropy(logits.reshape(-1,tokenizer.vocab_size),labels.reshape(-1))
        accumulated_loss+=float(loss.detach())
        scaler.scale(loss/GRADIENT_ACCUMULATION_STEPS).backward()
    current_lr=get_learning_rate(global_step)
    for group in optimizer.param_groups: group["lr"]=current_lr
    scaler.unscale_(optimizer)
    grad_norm=torch.nn.utils.clip_grad_norm_(model.parameters(),MAX_GRAD_NORM)
    scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
    global_step+=1; tokens_seen+=TOKENS_PER_STEP
    train_loss=accumulated_loss/GRADIENT_ACCUMULATION_STEPS
    if global_step==1 or global_step%LOG_EVERY_STEPS==0:
        print(f"Step {global_step:,}/{MAX_TRAIN_STEPS:,} | loss {train_loss:.4f} | lr {current_lr:.2e} | grad {float(grad_norm):.3f} | tokens {tokens_seen:,} | elapsed {(time.time()-start_time)/3600:.2f}h")
        history["step"].append(global_step); history["train_loss"].append(train_loss); history["learning_rate"].append(current_lr); history["tokens_seen"].append(tokens_seen)
    if global_step%EVAL_EVERY_STEPS==0 or global_step==MAX_TRAIN_STEPS:
        val_loss,val_ppl=evaluate_model(); history["val_loss"].append(val_loss); history["val_perplexity"].append(val_ppl)
        print(f"Validation sample | step {global_step} | loss {val_loss:.4f} | perplexity {val_ppl:.2f}")
        if val_loss<best_val_loss:
            best_val_loss=val_loss
            torch.save({"step":global_step,"model_state_dict":model.state_dict(),"optimizer_state_dict":optimizer.state_dict(),"best_val_loss":best_val_loss,"history":history},BEST_CHECKPOINT)
    if global_step%SAVE_EVERY_STEPS==0 or global_step==MAX_TRAIN_STEPS:
        torch.save({"step":global_step,"model_state_dict":model.state_dict(),"optimizer_state_dict":optimizer.state_dict(),"best_val_loss":best_val_loss,"history":history},LATEST_CHECKPOINT)
        with open(os.path.join(OUTPUT_DIR,"training_history.json"),"w",encoding="utf-8") as f: json.dump(history,f,indent=2)
        print("Checkpoint and history saved.")
print("V5 Phase 1 pretraining complete; best sampled validation loss:",best_val_loss)


In [ ]:
best_checkpoint=torch.load(BEST_CHECKPOINT,map_location=DEVICE,weights_only=False)
model.load_state_dict(best_checkpoint["model_state_dict"],strict=True); model.eval(); best_val_loss=best_checkpoint["best_val_loss"]
print("Restored best V5 checkpoint. step=",best_checkpoint["step"],"val loss=",best_val_loss)


## 12. Perplexity

In [ ]:
# Perplexity from the best validation checkpoint.

perplexity = math.exp(best_val_loss)

print(f"Validation loss: {best_val_loss:.6f}")
print(f"Validation perplexity: {perplexity:.4f}")

## 13. Plot Training and Validation Loss

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.plot(
    history["step"],
    history["train_loss"],
    label="Training Loss"
)

val_steps = [
    min(
        (i + 1) * EVAL_EVERY_STEPS,
        MAX_TRAIN_STEPS
    )
    for i in range(
        len(history["val_loss"])
    )
]

plt.plot(
    val_steps,
    history["val_loss"],
    marker="o",
    label="Validation Loss"
)

plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("V1 Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()

LOSS_PLOT_PATH = os.path.join(
    OUTPUT_DIR,
    "v1_loss_curve.png"
)

plt.savefig(
    LOSS_PLOT_PATH,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("Saved:", LOSS_PLOT_PATH)

## 14. Save Final Model to Kaggle Output

In [ ]:
final_model_path=os.path.join(MODEL_OUTPUT_DIR,"v5_pretrained.pt")
torch.save({"model_state_dict":model.state_dict(),"config":{"vocab_size":tokenizer.vocab_size,"n_embd":N_EMBD,"n_layer":N_LAYER,"n_head":N_HEAD,"n_kv_head":N_KV_HEAD,"ffn_multiplier":FFN_MULTIPLIER,"context_length":MODEL_CONTEXT_LENGTH},"actual_parameters":sum(p.numel() for p in model.parameters()),"best_val_loss":best_val_loss},final_model_path)
print("Saved V5 from-scratch pretrained checkpoint:",final_model_path)


## Phase 4 — CJPE task adaptation and knowledge distillation

**Inputs to attach to Kaggle:** `train.json`, `dev.json`, `test.json`, `bpe_en_25k.json`, and the Phase 1 `v1_model.pt`. The V5 teacher is produced earlier in this same notebook. Task records must be lists of `{id, text, label}` (test may omit `label`) with binary labels 0/1. The cells validate IDs, fields, and split separation before fitting.

**Experiment:** First fine-tune the Phase 1-pretrained V5 backbone plus a new binary head as a supervised teacher. Then freeze that selected teacher and train a student initialized from the existing Phase 1 V1 checkpoint. The student objective combines class-weighted supervised cross-entropy with temperature-scaled KL divergence from teacher probabilities. This isolates the effect of transferring task-specific teacher knowledge into a smaller student; it does not compare all PEFT methods in this single notebook.

**Duplicate-record handling:** the supplied train/dev files contain exact repeated case rows. The loader keeps one copy per ID, removes only rows with identical text and label, and raises an error if an ID has conflicting content or labels.


In [ ]:
# Locate required files exactly once; stop on ambiguous Kaggle attachments.
from pathlib import Path
from collections import Counter
import numpy as np, json, zipfile, gc
from torch.utils.data import Dataset,DataLoader
INPUT_ROOT=Path("/kaggle/input")
def find_one(name):
    found=sorted(p for p in INPUT_ROOT.rglob(name) if p.is_file())
    if len(found)!=1: raise FileNotFoundError(f"Expected exactly one {name} under /kaggle/input; found {len(found)}: {found[:6]}")
    return found[0]
TASK_PATHS={name:find_one(name) for name in ("train.json","dev.json","test.json","v1_model.pt")}
# Reuse BPE_PATH loaded in the Phase 1 section; require the exact tokenizer asset.
print("Phase 4 inputs:",TASK_PATHS)

def load_cases(path,labelled):
    rows=json.loads(Path(path).read_text(encoding="utf-8"))
    if not isinstance(rows,list): raise TypeError(f"{path}: expected a JSON list")
    result=[]; seen={}; duplicate_count=0
    for i,row in enumerate(rows):
        if not isinstance(row,dict) or "id" not in row or "text" not in row: raise ValueError(f"{path}: row {i} needs id and text")
        cid=str(row["id"])
        if not isinstance(row["text"],str): raise ValueError(f"{cid}: text must be a string")
        item={"id":cid,"text":row["text"]}
        if labelled:
            y=row.get("label")
            if type(y) is not int or y not in (0,1): raise ValueError(f"{cid}: label must be integer 0/1")
            item["label"]=y
        if cid in seen:
            if result[seen[cid]] != item: raise ValueError(f"{path}: conflicting duplicate id {cid} (text or label differs)")
            duplicate_count+=1
            continue
        seen[cid]=len(result); result.append(item)
    if duplicate_count: print(f"{path}: removed {duplicate_count} exact duplicate rows; kept {len(result)} unique IDs")
    return result
TRAIN_ROWS=load_cases(TASK_PATHS["train.json"],True); DEV_ROWS=load_cases(TASK_PATHS["dev.json"],True); TEST_ROWS=load_cases(TASK_PATHS["test.json"],False)
IDSETS=[{r["id"] for r in rows} for rows in (TRAIN_ROWS,DEV_ROWS,TEST_ROWS)]
if any(IDSETS[i]&IDSETS[j] for i in range(3) for j in range(i+1,3)): raise ValueError("Train/dev/test IDs overlap")
print("Rows:",len(TRAIN_ROWS),len(DEV_ROWS),len(TEST_ROWS),"| train labels:",dict(Counter(r["label"] for r in TRAIN_ROWS)))


### Windowing and aggregation

Judgments can exceed the model context. We encode at most six 512-token windows distributed across the full document, with a 384-token stride. Training rotates through those windows across epochs, limiting per-step cost while exposing different sections. Development and test aggregate all selected window logits by their mean. This is a compute/coverage compromise and is reported as such; it does not imply every token is processed.


In [ ]:
CHUNK_TOKENS=512; CHUNK_STRIDE=384; MAX_CHUNKS_PER_CASE=6; TASK_BATCH=2; TASK_ACCUM=4; INFER_BATCH=8

def make_windows(text):
    ids=tokenizer.encode(text,add_bos=True)
    if not ids: ids=[BOS_TOKEN_ID]
    if len(ids)<=CHUNK_TOKENS: starts=[0]
    else:
        starts_all=list(range(0,len(ids)-CHUNK_TOKENS+1,CHUNK_STRIDE))
        if starts_all[-1]!=len(ids)-CHUNK_TOKENS: starts_all.append(len(ids)-CHUNK_TOKENS)
        take=min(MAX_CHUNKS_PER_CASE,len(starts_all))
        starts=[starts_all[j] for j in sorted(set(np.linspace(0,len(starts_all)-1,take).round().astype(int)))]
    out=[]
    for start in starts:
        x=ids[start:start+CHUNK_TOKENS]
        out.append(x+[PAD_TOKEN_ID]*(CHUNK_TOKENS-len(x)))
    return np.asarray(out,dtype=np.int64)

def encode_split(rows,name):
    out=[]
    for i,row in enumerate(rows):
        out.append(make_windows(row["text"]))
        if (i+1)%2000==0: print(name,i+1,"/",len(rows))
    print(name,"documents",len(rows),"windows",sum(len(x) for x in out))
    return out
TRAIN_WINDOWS=encode_split(TRAIN_ROWS,"train"); DEV_WINDOWS=encode_split(DEV_ROWS,"dev"); TEST_WINDOWS=encode_split(TEST_ROWS,"test")
class CaseTrain(Dataset):
    def __init__(self,rows,windows): self.rows=rows; self.windows=windows; self.epoch=0
    def set_epoch(self,e): self.epoch=e
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        ws=self.windows[i]; ids=ws[(i+self.epoch)%len(ws)]
        return torch.as_tensor(ids,dtype=torch.long),self.rows[i]["label"]
train_ds=CaseTrain(TRAIN_ROWS,TRAIN_WINDOWS)
train_loader=DataLoader(train_ds,batch_size=TASK_BATCH,shuffle=True,num_workers=0,pin_memory=(DEVICE.type=="cuda"),drop_last=False)


### Task classifier and V5 teacher training

The representation is the last non-padding token of each window, followed by a dropout layer and a two-class linear head. The classifier is trained with class-weighted cross-entropy from the training split. Early stopping selects by development macro-F1, which gives both outcomes equal weight under class imbalance. The V5 backbone is fully fine-tuned here; this is the task teacher, not a PEFT comparison.


In [ ]:
PROGRESS_EVERY=50
class CaseClassifier(nn.Module):
    def __init__(self,backbone,width,dropout=0.1):
        super().__init__(); self.backbone=backbone; self.head=nn.Sequential(nn.Dropout(dropout),nn.Linear(width,2))
    def forward(self,input_ids):
        hidden=self.backbone(input_ids,return_hidden=True)
        mask=input_ids.ne(PAD_TOKEN_ID); last=(mask.sum(1)-1).clamp(min=0)
        pooled=hidden[torch.arange(hidden.size(0),device=hidden.device),last]
        return self.head(pooled)

def load_backbone(n_embd,n_layer,n_head,n_kv,context):
    return V1LegalLM(vocab_size=tokenizer.vocab_size,n_embd=n_embd,n_layer=n_layer,n_head=n_head,n_kv_head=n_kv,max_seq_len=context,ffn_multiplier=FFN_MULTIPLIER,dropout=0.0)

def macro_metrics(y,p):
    f=[]
    for c in (0,1):
        tp=sum(a==c and b==c for a,b in zip(y,p)); fp=sum(a!=c and b==c for a,b in zip(y,p)); fn=sum(a==c and b!=c for a,b in zip(y,p))
        pr=tp/(tp+fp) if tp+fp else 0.; re=tp/(tp+fn) if tp+fn else 0.; f.append(2*pr*re/(pr+re) if pr+re else 0.)
    return {"macro_f1":sum(f)/2,"f1_class_0":f[0],"f1_class_1":f[1],"accuracy":sum(a==b for a,b in zip(y,p))/max(1,len(y))}

@torch.no_grad()
def predict_cases(model,rows,windows):
    model.eval(); flat=[]; owners=[]
    for i,ws in enumerate(windows):
        flat.extend(ws); owners.extend([i]*len(ws))
    sums=np.zeros((len(rows),2),np.float64); counts=np.zeros(len(rows),np.int64)
    amp_dtype=torch.bfloat16 if DEVICE.type=="cuda" and torch.cuda.is_bf16_supported() else torch.float16
    infer_started=time.time(); infer_total=math.ceil(len(flat)/INFER_BATCH)
    for batch_idx,j in enumerate(range(0,len(flat),INFER_BATCH),start=1):
        x=torch.as_tensor(np.asarray(flat[j:j+INFER_BATCH]),dtype=torch.long,device=DEVICE)
        with torch.autocast(device_type="cuda",dtype=amp_dtype,enabled=USE_AMP): z=model(x)
        z=z.float().cpu().numpy()
        for k,v in enumerate(z): sums[owners[j+k]]+=v; counts[owners[j+k]]+=1
        if batch_idx==1 or batch_idx%PROGRESS_EVERY==0 or batch_idx==infer_total:
            elapsed=time.time()-infer_started; rate=batch_idx/max(elapsed,1e-6); eta=(infer_total-batch_idx)/max(rate,1e-6)
            print(f"Predict cases: batch {batch_idx}/{infer_total} | elapsed {elapsed/60:.1f} min | ETA {eta/60:.1f} min",flush=True)
    return (sums/counts[:,None]).astype(np.float32)

counts=np.bincount([r["label"] for r in TRAIN_ROWS],minlength=2)
CLASS_WEIGHTS=torch.tensor(len(TRAIN_ROWS)/(2*np.maximum(counts,1)),dtype=torch.float32,device=DEVICE)


In [ ]:
# 9. V1 Decoder-Only Transformer
# This is our first architecture built from scratch.
# V1 uses:
# - V1RMSNorm
# - RoPE
# - Grouped Query Attention
# - Causal self-attention
# - Standard GELU FFN
# - Residual connections
# - KV cache for autoregressive generation

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


# ---------------------------------------------------------
# V1RMSNorm
# ---------------------------------------------------------

class V1RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(
            dim=-1,
            keepdim=True
        )

        x = x * torch.rsqrt(
            rms + self.eps
        )

        return self.weight * x


# ---------------------------------------------------------
# Rotary Positional Embeddings
# ---------------------------------------------------------

class V1RotaryEmbedding(nn.Module):
    def __init__(
        self,
        head_dim,
        max_seq_len,
        base=10000
    ):
        super().__init__()

        self.head_dim = head_dim
        self.max_seq_len = max_seq_len

        inv_freq = 1.0 / (
            base ** (
                torch.arange(
                    0,
                    head_dim,
                    2
                ).float()
                / head_dim
            )
        )

        self.register_buffer(
            "inv_freq",
            inv_freq,
            persistent=False
        )

        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        positions = torch.arange(
            seq_len,
            device=self.inv_freq.device
        ).float()

        freqs = torch.outer(
            positions,
            self.inv_freq
        )

        self.register_buffer(
            "cos_cached",
            freqs.cos(),
            persistent=False
        )

        self.register_buffer(
            "sin_cached",
            freqs.sin(),
            persistent=False
        )

    def forward(
        self,
        q,
        k,
        position_offset=0
    ):
        seq_len = q.size(-2)

        required_len = (
            position_offset + seq_len
        )

        if required_len > self.max_seq_len:
            self._build_cache(required_len)
            self.max_seq_len = required_len

        cos = self.cos_cached[
            position_offset:
            position_offset + seq_len
        ]

        sin = self.sin_cached[
            position_offset:
            position_offset + seq_len
        ]

        cos = cos.unsqueeze(0).unsqueeze(0)
        sin = sin.unsqueeze(0).unsqueeze(0)

        q = self._apply_rotary(
            q,
            cos,
            sin
        )

        k = self._apply_rotary(
            k,
            cos,
            sin
        )

        return q, k

    @staticmethod
    def _apply_rotary(x, cos, sin):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]

        rotated = torch.stack(
            (-x2, x1),
            dim=-1
        ).flatten(-2)

        cos_full = torch.repeat_interleave(
            cos,
            2,
            dim=-1
        )

        sin_full = torch.repeat_interleave(
            sin,
            2,
            dim=-1
        )

        return (
            x * cos_full
            + rotated * sin_full
        )


# ---------------------------------------------------------
# Grouped Query Attention
# ---------------------------------------------------------

class V1GQAAttention(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        n_kv_head,
        max_seq_len,
        dropout=0.0
    ):
        super().__init__()

        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0

        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.head_dim = n_embd // n_head

        self.kv_repeat = (
            n_head // n_kv_head
        )

        self.q_proj = nn.Linear(
            n_embd,
            n_head * self.head_dim,
            bias=False
        )

        self.k_proj = nn.Linear(
            n_embd,
            n_kv_head * self.head_dim,
            bias=False
        )

        self.v_proj = nn.Linear(
            n_embd,
            n_kv_head * self.head_dim,
            bias=False
        )

        self.out_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.rope = V1RotaryEmbedding(
            head_dim=self.head_dim,
            max_seq_len=max_seq_len
        )

        self.dropout = dropout

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False
    ):
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B,
            T,
            self.n_head,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.n_kv_head,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.n_kv_head,
            self.head_dim
        ).transpose(1, 2)

        past_len = 0

        if past_key_value is not None:
            past_k, past_v = past_key_value
            past_len = past_k.size(-2)

        # Apply RoPE to the new Q and K positions.
        q, k = self.rope(
            q,
            k,
            position_offset=past_len
        )

        # Add the new K/V to the cached K/V during generation.
        if past_key_value is not None:
            k = torch.cat(
                [past_k, k],
                dim=-2
            )

            v = torch.cat(
                [past_v, v],
                dim=-2
            )

        present = None

        if use_cache:
            present = (k, v)

        # GQA: each KV head is shared by a group of query heads.
        if self.kv_repeat > 1:
            k = k.repeat_interleave(
                self.kv_repeat,
                dim=1
            )

            v = v.repeat_interleave(
                self.kv_repeat,
                dim=1
            )

        # During training, each token can only attend to
        # itself and tokens before it.
        #
        # IMPORTANT:
        # For scaled_dot_product_attention, True means
        # "this position is allowed to attend".
        if past_key_value is None:
            attn_mask = torch.tril(
                torch.ones(
                    T,
                    T,
                    device=x.device,
                    dtype=torch.bool
                )
            )
        else:
            # During cached decoding, all cached positions
            # are valid keys for the current token.
            attn_mask = None

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=(
                self.dropout
                if self.training
                else 0.0
            ),
            is_causal=False
        )

        y = y.transpose(
            1,
            2
        ).contiguous()

        y = y.view(
            B,
            T,
            C
        )

        y = self.out_proj(y)

        return y, present


# ---------------------------------------------------------
# Standard FFN
# ---------------------------------------------------------

class V1FeedForward(nn.Module):
    def __init__(
        self,
        n_embd,
        multiplier=4,
        dropout=0.0
    ):
        super().__init__()

        hidden_dim = int(
            n_embd * multiplier
        )

        self.fc1 = nn.Linear(
            n_embd,
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            n_embd
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        x = self.dropout(x)

        return x


# ---------------------------------------------------------
# Transformer Block
# ---------------------------------------------------------

class V1TransformerBlock(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        n_kv_head,
        max_seq_len,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.norm1 = V1RMSNorm(n_embd)

        self.attention = V1GQAAttention(
            n_embd=n_embd,
            n_head=n_head,
            n_kv_head=n_kv_head,
            max_seq_len=max_seq_len,
            dropout=dropout
        )

        self.norm2 = V1RMSNorm(n_embd)

        self.ffn = V1FeedForward(
            n_embd=n_embd,
            multiplier=ffn_multiplier,
            dropout=dropout
        )

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False
    ):
        # Pre-norm attention followed by a residual connection.
        attn_output, present = self.attention(
            self.norm1(x),
            past_key_value=past_key_value,
            use_cache=use_cache
        )

        x = x + attn_output

        # Pre-norm FFN followed by a residual connection.
        x = x + self.ffn(
            self.norm2(x)
        )

        return x, present


# ---------------------------------------------------------
# V1 Language Model
# ---------------------------------------------------------

class V1CheckpointLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embd,
        n_layer,
        n_head,
        n_kv_head,
        max_seq_len,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.n_embd = n_embd

        self.token_embedding = nn.Embedding(
            vocab_size,
            n_embd
        )

        self.blocks = nn.ModuleList([
            V1TransformerBlock(
                n_embd=n_embd,
                n_head=n_head,
                n_kv_head=n_kv_head,
                max_seq_len=max_seq_len,
                ffn_multiplier=ffn_multiplier,
                dropout=dropout
            )
            for _ in range(n_layer)
        ])

        self.final_norm = V1RMSNorm(
            n_embd
        )

        # Tie the LM head to the token embedding weights.
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False
        )

        self.lm_head.weight = (
            self.token_embedding.weight
        )

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False,
        return_hidden=False
    ):
        x = self.token_embedding(
            input_ids
        )

        presents = [] if use_cache else None

        for layer_index, block in enumerate(
            self.blocks
        ):
            past = None

            if past_key_values is not None:
                past = past_key_values[
                    layer_index
                ]

            if self.training and getattr(self, "gradient_checkpointing", False) and not use_cache:
                x = checkpoint(lambda h, layer=block: layer(h, past_key_value=None, use_cache=False)[0], x, use_reentrant=False)
                present = None
            else:
                x, present = block(
                    x,
                    past_key_value=past,
                    use_cache=use_cache
                )

            if use_cache:
                presents.append(
                    present
                )

        x = self.final_norm(x)
        if return_hidden:
            return x
        logits = self.lm_head(x)
        return logits, presents

In [ ]:
# Restore the trained-from-scratch V5 teacher backbone and attach a fresh Phase 4 head.
teacher_backbone=load_backbone(N_EMBD,N_LAYER,N_HEAD,N_KV_HEAD,MODEL_CONTEXT_LENGTH)
teacher_ckpt=torch.load(final_model_path,map_location="cpu",weights_only=False)
missing,unexpected=teacher_backbone.load_state_dict(teacher_ckpt["model_state_dict"],strict=True)
assert not missing and not unexpected
teacher_backbone.gradient_checkpointing=True
teacher=CaseClassifier(teacher_backbone,N_EMBD).to(DEVICE)
print("V5 teacher params:",sum(p.numel() for p in teacher.parameters()),"| trainable:",sum(p.numel() for p in teacher.parameters() if p.requires_grad))

def train_supervised_teacher(model,max_epochs=4,patience=2):
    opt=torch.optim.AdamW([{"params":model.backbone.parameters(),"lr":1e-5},{"params":model.head.parameters(),"lr":1e-4}],weight_decay=0.01)
    scaler=torch.amp.GradScaler("cuda",enabled=(USE_AMP and not torch.cuda.is_bf16_supported()))
    best=-1.; best_state=None; stale=0; hist=[]
    for epoch in range(max_epochs):
        train_ds.set_epoch(epoch); model.train(); opt.zero_grad(set_to_none=True); losses=[]; epoch_started=time.time()
        print(f"Starting Teacher epoch {epoch+1}/{max_epochs}: {len(train_loader)} batches",flush=True)
        for step,(ids,y) in enumerate(train_loader):
            ids=ids.to(DEVICE); y=y.to(DEVICE); amp_dtype=torch.bfloat16 if DEVICE.type=="cuda" and torch.cuda.is_bf16_supported() else torch.float16
            with torch.autocast(device_type="cuda",dtype=amp_dtype,enabled=USE_AMP): logits=model(ids); loss=F.cross_entropy(logits.float(),y,weight=CLASS_WEIGHTS)
            accum_now=min(TASK_ACCUM,len(train_loader)-(step//TASK_ACCUM)*TASK_ACCUM)
            scaler.scale(loss/accum_now).backward(); losses.append(float(loss.detach()))
            if step==0 or (step+1)%PROGRESS_EVERY==0 or step+1==len(train_loader):
                elapsed=time.time()-epoch_started; rate=(step+1)/max(elapsed,1e-6); eta=(len(train_loader)-step-1)/max(rate,1e-6)
                print(f"Teacher epoch {epoch+1}/{max_epochs}: batch {step+1}/{len(train_loader)} | running loss {float(np.mean(losses)):.4f} | elapsed {elapsed/60:.1f} min | ETA {eta/60:.1f} min",flush=True)
            if (step+1)%TASK_ACCUM==0 or step+1==len(train_loader):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        dev_logits=predict_cases(model,DEV_ROWS,DEV_WINDOWS); dev_y=[r["label"] for r in DEV_ROWS]; dev_pred=(dev_logits[:,1]>dev_logits[:,0]).astype(int).tolist(); metrics=macro_metrics(dev_y,dev_pred)
        hist.append({"epoch":epoch+1,"train_loss":float(np.mean(losses)),**metrics}); print("Teacher epoch",epoch+1,"train loss",round(float(np.mean(losses)),4),metrics)
        if metrics["macro_f1"]>best:
            best=metrics["macro_f1"]; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; stale=0
            torch.save({"state_dict":best_state,"epoch":epoch+1,"dev_metrics":metrics},OUT/"v5_phase4_teacher_best.pt")
        else: stale+=1
        if stale>=patience: break
    model.load_state_dict(best_state); return hist

teacher_history=train_supervised_teacher(teacher)
(OUT/"teacher_training_history.json").write_text(json.dumps(teacher_history,indent=2),encoding="utf-8")


### Distill the V5 task teacher into the V1 student

The teacher is frozen. The student starts from `v1_model.pt`, loaded strictly into the same V1 architecture used in Phase 1 (12 layers, width 768, 12 query heads, 4 KV heads). Each batch uses the same sampled window for both models. The loss is `0.5 × weighted CE + 0.5 × T² × KL`, with `T=2`; CE preserves the gold label while KL transfers the teacher’s softened class preference. This makes the experiment measurable against ordinary supervised learning rather than assuming distillation helps.


In [ ]:
# Freeze the selected teacher and restore its best development checkpoint.
teacher_best=torch.load(OUT/"v5_phase4_teacher_best.pt",map_location=DEVICE,weights_only=False)
teacher.load_state_dict(teacher_best["state_dict"],strict=True); teacher.eval()
for p in teacher.parameters(): p.requires_grad_(False)
# Load the smaller V1 student from the existing Phase 1 checkpoint.
student_backbone=V1CheckpointLM(vocab_size=tokenizer.vocab_size,n_embd=768,n_layer=12,n_head=12,n_kv_head=4,max_seq_len=1024,ffn_multiplier=4,dropout=0.0)
v1_ckpt=torch.load(TASK_PATHS["v1_model.pt"],map_location="cpu",weights_only=False)
v1_state=v1_ckpt.get("model_state_dict",v1_ckpt)
student_backbone.load_state_dict(v1_state,strict=True)
student_backbone.gradient_checkpointing=True
student=CaseClassifier(student_backbone,768).to(DEVICE)
print("Student total params:",sum(p.numel() for p in student.parameters()),"| trainable:",sum(p.numel() for p in student.parameters() if p.requires_grad))
TEMPERATURE=2.0; KD_ALPHA=0.5

def train_student_kd(max_epochs=5,patience=2):
    opt=torch.optim.AdamW(student.parameters(),lr=8e-6,weight_decay=0.01)
    scaler=torch.amp.GradScaler("cuda",enabled=(USE_AMP and not torch.cuda.is_bf16_supported()))
    best=-1.; best_state=None; stale=0; hist=[]
    for epoch in range(max_epochs):
        train_ds.set_epoch(epoch); student.train(); opt.zero_grad(set_to_none=True); losses=[]; epoch_started=time.time()
        print(f"Starting KD student epoch {epoch+1}/{max_epochs}: {len(train_loader)} batches",flush=True)
        for step,(ids,y) in enumerate(train_loader):
            ids=ids.to(DEVICE); y=y.to(DEVICE); amp_dtype=torch.bfloat16 if DEVICE.type=="cuda" and torch.cuda.is_bf16_supported() else torch.float16
            with torch.no_grad(),torch.autocast(device_type="cuda",dtype=amp_dtype,enabled=USE_AMP): teacher_logits=teacher(ids).float()
            with torch.autocast(device_type="cuda",dtype=amp_dtype,enabled=USE_AMP): student_logits=student(ids).float()
            ce=F.cross_entropy(student_logits,y,weight=CLASS_WEIGHTS)
            kl=F.kl_div(F.log_softmax(student_logits/TEMPERATURE,dim=-1),F.softmax(teacher_logits/TEMPERATURE,dim=-1),reduction="batchmean")*(TEMPERATURE**2)
            loss=(1-KD_ALPHA)*ce+KD_ALPHA*kl
            accum_now=min(TASK_ACCUM,len(train_loader)-(step//TASK_ACCUM)*TASK_ACCUM)
            scaler.scale(loss/accum_now).backward(); losses.append(float(loss.detach()))
            if step==0 or (step+1)%PROGRESS_EVERY==0 or step+1==len(train_loader):
                elapsed=time.time()-epoch_started; rate=(step+1)/max(elapsed,1e-6); eta=(len(train_loader)-step-1)/max(rate,1e-6)
                print(f"KD student epoch {epoch+1}/{max_epochs}: batch {step+1}/{len(train_loader)} | running loss {float(np.mean(losses)):.4f} | elapsed {elapsed/60:.1f} min | ETA {eta/60:.1f} min",flush=True)
            if (step+1)%TASK_ACCUM==0 or step+1==len(train_loader):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(student.parameters(),1.0); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        dev_logits=predict_cases(student,DEV_ROWS,DEV_WINDOWS); ydev=[r["label"] for r in DEV_ROWS]; pred=(dev_logits[:,1]>dev_logits[:,0]).astype(int).tolist(); metrics=macro_metrics(ydev,pred)
        hist.append({"epoch":epoch+1,"train_loss":float(np.mean(losses)),**metrics}); print("KD student epoch",epoch+1,"loss",round(float(np.mean(losses)),4),metrics)
        if metrics["macro_f1"]>best:
            best=metrics["macro_f1"]; best_state={k:v.detach().cpu().clone() for k,v in student.state_dict().items()}; stale=0
            torch.save({"state_dict":best_state,"epoch":epoch+1,"dev_metrics":metrics},OUT/"v1_kd_student_best.pt")
        else: stale+=1
        if stale>=patience: break
    student.load_state_dict(best_state); return hist

student_history=train_student_kd()
(OUT/"student_kd_training_history.json").write_text(json.dumps(student_history,indent=2),encoding="utf-8")


### Development-only threshold selection and Codabench files

Each model selects a binary threshold from its development probabilities; the threshold is then fixed for test prediction. The two submissions are packaged separately so results can be compared without mixing model outputs. Each ZIP contains exactly one root-level `prediction.json` with `{id, pred}` entries. This notebook validates lengths, order, label values, ZIP members, and CRC before it reports the paths. Scores must be measured by the official evaluator after submission.


In [ ]:
def choose_threshold(logits,rows):
    probs=torch.softmax(torch.tensor(logits),dim=-1).numpy()[:,1]; y=[r["label"] for r in rows]
    trials=[]
    for t in np.arange(0.30,0.701,0.01):
        p=(probs>=t).astype(int).tolist(); m=macro_metrics(y,p); trials.append((m["macro_f1"],-abs(float(t)-0.5),float(t),m))
    _,_,threshold,metrics=max(trials,key=lambda x:(x[0],x[1]))
    return threshold,metrics

def write_submission(model,name):
    dev_logits=predict_cases(model,DEV_ROWS,DEV_WINDOWS); threshold,dev_metrics=choose_threshold(dev_logits,DEV_ROWS)
    test_logits=predict_cases(model,TEST_ROWS,TEST_WINDOWS); probs=torch.softmax(torch.tensor(test_logits),dim=-1).numpy()[:,1]
    preds=(probs>=threshold).astype(int).tolist()
    payload=[{"id":r["id"],"pred":int(y)} for r,y in zip(TEST_ROWS,preds)]
    assert len(payload)==len(TEST_ROWS) and [x["id"] for x in payload]==[r["id"] for r in TEST_ROWS]
    assert all(type(x["pred"]) is int and x["pred"] in (0,1) for x in payload)
    pred_path=OUT/f"{name}_prediction.json"; pred_path.write_text(json.dumps(payload,ensure_ascii=False,indent=2),encoding="utf-8")
    zip_path=OUT/f"{name}_submission.zip"
    with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as z: z.write(pred_path,arcname="prediction.json")
    with zipfile.ZipFile(zip_path) as z:
        assert z.namelist()==["prediction.json"] and z.testzip() is None
        assert json.loads(z.read("prediction.json"))==payload
    return {"threshold":threshold,"dev_metrics":dev_metrics,"prediction_count":len(payload),"zip":str(zip_path)}

teacher_result=write_submission(teacher,"v5_teacher")
student_result=write_submission(student,"v1_kd_student")
summary={"teacher":teacher_result,"student":student_result,"teacher_best_epoch":teacher_best["epoch"],"student_best_epoch":torch.load(OUT/"v1_kd_student_best.pt",map_location="cpu",weights_only=False)["epoch"],"teacher_parameters":sum(p.numel() for p in teacher.parameters()),"student_parameters":sum(p.numel() for p in student.parameters()),"kd_temperature":TEMPERATURE,"kd_alpha":KD_ALPHA}
(OUT/"phase4_comparison.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
print(json.dumps(summary,indent=2))


## Interpretation and limitations

The actual parameter count printed by the model is the authoritative count; the earlier notebook’s loose 50–300M assertion did not establish a 130M model. This revision requires the instantiated count to exceed 130M. The teacher’s development macro-F1 and the distilled student’s development macro-F1 are research measurements, not leaderboard scores. Distillation can underperform the teacher or even a supervised-only student; compare the two outputs and retain the better validated submission. This notebook does not yet include LoRA, LoRMA, or BitFit as separate Phase 4 runs.
